[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/littlekg87/lecture/blob/main/2025_kmooc/notebooks/04_wordcloud.ipynb)


# 단어 분석 ② — 김상헌 상소문 워드클라우드

단어 빈도를 **그림**으로 그려 봅니다.

**왼쪽의 ▶ 버튼을 위에서부터 차례대로 누르기만 하면 됩니다.**

> 💡 **왜 Colab을 쓰나요?**
> 강의 코드의 `font_path="malgun.ttf"` 는 윈도우 전용 폰트라 **맥에서는 오류**가 납니다.
> Colab에서는 나눔고딕을 설치해 운영체제와 상관없이 똑같은 결과가 나옵니다.


## 0단계 — 준비하기 (설치)

형태소 분석기·워드클라우드·**한글 폰트**를 한 번에 설치합니다.


In [ ]:
# 1~2분 걸립니다.
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null
!apt-get install -y -qq fonts-nanum > /dev/null   # <- 한글 폰트 (맑은 고딕 대신)
!pip install -q konlpy wordcloud
!fc-cache -f > /dev/null

import glob, os

jvm = sorted(glob.glob('/usr/lib/jvm/java-*-openjdk-amd64'))
os.environ['JAVA_HOME'] = jvm[-1]

# 설치된 한글 폰트 경로를 찾아 둡니다
FONT_PATH = glob.glob('/usr/share/fonts/**/NanumGothic.ttf', recursive=True)[0]
print('한글 폰트:', FONT_PATH)
print('설치 완료!')


## 0단계 — 실습 데이터 내려받기


In [ ]:
!wget -q -O kim-sangheon-sangso.txt "https://raw.githubusercontent.com/littlekg87/lecture/main/2025_kmooc/data/word-analysis/kim-sangheon-sangso.txt"

print('내려받기 완료!')

# (선택) 내 컴퓨터의 다른 텍스트 파일로 해보고 싶다면
# 아래 두 줄의 # 을 지우고 실행한 뒤 파일을 선택하세요.
# from google.colab import files
# files.upload()


## 1~4단계 — 텍스트 읽기 → 명사 추출 → 불용어 제거 → 빈도 계산

앞의 ③ 실습과 같은 과정입니다. 한 칸에 모았습니다.


In [ ]:
import re
from collections import Counter
from konlpy.tag import Okt

# 텍스트 파일 읽기
with open('kim-sangheon-sangso.txt', 'r', encoding='utf-8') as file:
    text = file.read()

# 한글 이외의 문자 제거
text = re.sub(r'[^가-힣\s]', '', text)

# 형태소 분석 (명사 추출)
okt = Okt()
tokens = okt.nouns(text)

# 불용어 제거
stopwords = ['것', '저', '그', '이', '수', '있다', '하다']
tokens = [w for w in tokens if w not in stopwords and len(w) > 1]

# 단어 빈도수 계산
word_counts = Counter(tokens)

print(f'단어 {len(word_counts):,}종류')
print(word_counts.most_common(20))


## 5단계 — 워드클라우드 만들기

`font_path` 에 위에서 찾아 둔 나눔고딕 경로를 넣는 것이 **한글이 깨지지 않는 핵심**입니다.


In [ ]:
from wordcloud import WordCloud

wordcloud = WordCloud(
    font_path=FONT_PATH,      # <- 윈도우 전용 'malgun.ttf' 대신 이걸 씁니다
    background_color='white',
    width=800,
    height=600
).generate_from_frequencies(word_counts)

print('워드클라우드 생성 완료!')


## 6단계 — 화면에 그리기


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')  # 축 제거
plt.show()


## 7단계 — 그림 파일로 내려받기


In [ ]:
wordcloud.to_file('wordcloud.png')

from google.colab import files
files.download('wordcloud.png')


## 더 해보기 — 모양 바꾸기

아래 값을 바꿔 가며 여러 번 실행해 보세요.

| 옵션 | 뜻 | 예시 |
|---|---|---|
| `background_color` | 배경색 | `'white'`, `'black'` |
| `max_words` | 보여줄 단어 수 | `100`, `50` |
| `colormap` | 색조합 | `'viridis'`, `'Blues'`, `'autumn'` |
| `width` / `height` | 크기 | `1600` / `1200` |


In [ ]:
wordcloud2 = WordCloud(
    font_path=FONT_PATH,
    background_color='black',
    colormap='autumn',
    max_words=80,
    width=1600,
    height=1200
).generate_from_frequencies(word_counts)

plt.figure(figsize=(12, 9))
plt.imshow(wordcloud2, interpolation='bilinear')
plt.axis('off')
plt.show()
